In [6]:
import pandas as pd
import arcticdb as adb
import nasdaqdatalink as ndl
%matplotlib inline
import time
from tqdm import tqdm

In [7]:
ndl.ApiConfig.api_key = "9yUyP5fu4aWYHRXYAqxx"

In [10]:
START = "2000-01-01"
END = "2026-02-15"

ARCTIC_URI = "lmdb://./database/arctic"

STOCK_LIBRARY = "stocks_lib"
ETF_LIBRARY = "etfs_lib"
REMOVED_LIB = "removed_stocks"
ac = adb.Arctic(ARCTIC_URI)

20260522 11:27:42.466560 867155 W arcticdb | LMDB path at /home/sysadmin/PycharmProjects/PM/database/arctic/ has already been opened in this process which is not supported by LMDB. You should only open a single Arctic instance over a given LMDB path. To continue safely, you should delete this Arctic instance and any others over the LMDB path in this process and then try again. Current process ID=[867155]


In [11]:
for lib in [STOCK_LIBRARY, ETF_LIBRARY, REMOVED_LIB]:
    if lib not in ac.list_libraries():
        ac.create_library(lib)

etf_lib = ac["etfs_lib"]
stocks_lib = ac["stocks_lib"]
removed_lib = ac["removed_stocks"]

In [12]:
stocks = ndl.get_table("SHARADAR/TICKERS", paginate=True)


In [13]:
stocks

,table,permaticker,ticker,name,exchange,isdelisted,category,cusips,siccode,sicsector,...,currency,location,lastupdated,firstadded,firstpricedate,lastpricedate,firstquarter,lastquarter,secfilings,companysite
None,,,,,,,,,,,,,,,,,,,,,
0,SFP,6399208,BFLX,ISHARES FLEXIBLE EQUITY ACTIVE ETF,NASDAQ,N,ETF,None,NaN,None,...,USD,Delaware; U.S.A,2026-05-21,2026-05-20,2026-05-20,2026-05-21,None,None,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
1,SFP,6399204,CVGD,CRESALTA GLOBAL DIVIDEND ETF,NYSEARCA,N,ETF,None,NaN,None,...,USD,Colorado; U.S.A,2026-05-21,2026-05-21,2026-05-18,2026-05-21,None,None,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
2,SFP,6399202,PLA,GRANITESHARES AUTOCALLABLE PLTR ETF,NASDAQ,N,ETF,None,NaN,None,...,USD,New York; U.S.A,2026-05-21,2026-05-20,2026-05-19,2026-05-21,None,None,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
3,SFP,6399197,LEND,SEI HIGH YIELD BOND & ALTERNATIVE CREDIT ETF,NASDAQ,N,ETF,None,NaN,None,...,USD,Pennsylvania; U.S.A,2026-05-21,2026-05-20,2026-05-18,2026-05-21,None,None,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
4,SFP,6399195,CVSM,CRESALTA SMALL AND MID-CAP ETF,NYSEARCA,N,ETF,None,NaN,None,...,USD,Colorado; U.S.A,2026-05-21,2026-05-21,2026-05-18,2026-05-21,None,None,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61896,SEP,101501,BBUCQ,BIG BUCK BREWERY & STEAKHOUSE INC,NASDAQ,Y,Domestic Common Stock,089072102 594194201 089072201 089072300,5810.0,Retail Trade,...,USD,Michigan; U.S.A,2021-08-28,2020-06-10,1996-06-13,2021-08-27,1996-12-31,2003-09-30,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
61897,SEP,101484,NWGN1,NEWGEN RESULTS CORP,NASDAQ,Y,Domestic Common Stock,651359101,7389.0,Services,...,USD,California; U.S.A,2020-08-03,2020-08-03,1999-05-21,2000-12-20,1997-12-31,2000-09-30,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
61898,SEP,101470,CYRS1,CYBERSENTRY INC,NYSEMKT,Y,Domestic Common Stock,23252H109,4899.0,Transportation Communications Electric Gas And...,...,USD,Florida; U.S.A,2024-04-01,2024-04-01,2000-01-26,2000-09-07,1997-12-31,2000-06-30,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None


In [14]:
print(stocks["exchange"].unique())
exchange_list = stocks["exchange"].unique()
print(exchange_list)
cols = ["table", "permaticker", "ticker", "isdelisted", "sicsector" , "firstpricedate" , "lastpricedate"]
all_tickers = stocks.loc[stocks["table"] == "SEP", cols]
etfs = stocks.loc[stocks["table"] == "SFP", cols]


['NASDAQ' 'NYSEARCA' 'BATS' 'NYSE' 'NYSEMKT' 'INDEX' None 'OTC']
['NASDAQ' 'NYSEARCA' 'BATS' 'NYSE' 'NYSEMKT' 'INDEX' None 'OTC']


Defensive just in case in the histor an etf and the stock is having the same name so that we won't get the data mixed for that case
it is rare but might happen since it is a long history

In [15]:
stock_to_perma = dict(zip(
    all_tickers["ticker"],
    all_tickers["permaticker"].astype("int64"),
))
etf_to_perma = dict(zip(
    etfs["ticker"],
    etfs["permaticker"].astype("int64"),
))
etf_to_perma

{'BFLX': 6399208,
 'CVGD': 6399204,
 'PLA': 6399202,
 'LEND': 6399197,
 'CVSM': 6399195,
 'HALX': 6399194,
 'AHD': 6399192,
 'PCSG': 6399188,
 'DLMY': 6399184,
 'PCLC': 6399181,
 'HONG': 6399176,
 'SMCP': 6399173,
 'WDRN': 6399170,
 'MSR': 6399168,
 'AVAZ': 6399167,
 'LOHA': 6399166,
 'NYYY': 6399165,
 'ALDB': 6399163,
 'GTND': 6399162,
 'BHYP': 6399161,
 'NISM': 6399160,
 'TYYY': 6399159,
 'SSCP': 6399156,
 'CBRG': 6399153,
 'LQID': 6399151,
 'GOVM': 6399150,
 'SUIL': 6399148,
 'TKNS': 6399111,
 'AMAU': 6399079,
 'ATC': 6399078,
 'CIEG': 6399077,
 'ETNG': 6399076,
 'STXU': 6399075,
 'SNDG': 6399074,
 'COHH': 6399073,
 'THYP': 6399072,
 'VAIE': 6399071,
 'AAOG': 6399070,
 'CATG': 6399069,
 'QVOL': 6399068,
 'BOT': 6399055,
 'IDMY': 6399038,
 'DOCK': 6399037,
 'DIPR': 6399035,
 'BUYB': 6399034,
 'BZZ': 6399033,
 'ODDZ': 6399032,
 'HMAY': 6399031,
 'JUDB': 6399030,
 'CLUB': 6399029,
 'TCAN': 6399028,
 'GNMX': 6399027,
 'CLOO': 6399026,
 'CTMA': 6399025,
 'EYES': 6399024,
 'BAY': 6399023,

### Checkpoint: there shouldn't be any duplicates.

In [24]:

dupes = all_tickers[all_tickers["ticker"].duplicated(keep=False)].sort_values("ticker")
print(f"{all_tickers['ticker'].duplicated().sum()} duplicate ticker rows")
dupes.head(20)


0 duplicate ticker rows


,table,permaticker,ticker,isdelisted,sicsector,firstpricedate,lastpricedate
None,,,,,,,


In [25]:
CHUNK_SIZE = 10
RETRY_ATTEMPTS = 3
DELAY_BETWEEN_CHUNKS = 0.5

#### reading off of S&P 500 data from sharadar, I am taking the name of all tickers historically in S&P500 and write them to arctic libs


In [26]:
sp500 = ndl.get_table("SHARADAR/SP500", paginate=True)
print(sp500.head(20))

           date   action ticker                              name  \
None                                                                
0    2026-05-21  current    ZTS                        ZOETIS INC   
1    2026-05-21  current   ZBRA           ZEBRA TECHNOLOGIES CORP   
2    2026-05-21  current    ZBH        ZIMMER BIOMET HOLDINGS INC   
3    2026-05-21  current    YUM                    YUM BRANDS INC   
4    2026-05-21  current    XYZ                         BLOCK INC   
5    2026-05-21  current    XYL                         XYLEM INC   
6    2026-05-21  current    XOM                  EXXON MOBIL CORP   
7    2026-05-21  current    XEL                   XCEL ENERGY INC   
8    2026-05-21  current   WYNN                  WYNN RESORTS LTD   
9    2026-05-21  current     WY                   WEYERHAEUSER CO   
10   2026-05-21  current    WTW          WILLIS TOWERS WATSON PLC   
11   2026-05-21  current    WST  WEST PHARMACEUTICAL SERVICES INC   
12   2026-05-21  current    WSM   

In [27]:
sp500["action"].value_counts()

action
historical    56690
added          1231
removed         734
current         503
Name: count, dtype: int64

In [28]:
ever_in_sp500 = sorted(
      sp500.loc[sp500["action"].isin(["current", "added", "removed"]), "ticker"].unique()
  )
len(ever_in_sp500)

1198

In [29]:
current_members = sp500.loc[sp500["action"] == "current", "ticker"].to_list()
print(current_members)

['ZTS', 'ZBRA', 'ZBH', 'YUM', 'XYZ', 'XYL', 'XOM', 'XEL', 'WYNN', 'WY', 'WTW', 'WST', 'WSM', 'WRB', 'WMT', 'WMB', 'WM', 'WFC', 'WELL', 'WEC', 'WDC', 'WDAY', 'WBD', 'WAT', 'WAB', 'VZ', 'VTRS', 'VTR', 'VST', 'VRTX', 'VRT', 'VRSN', 'VRSK', 'VMC', 'VLTO', 'VLO', 'VICI', 'VEEV', 'V', 'USB', 'URI', 'UPS', 'UNP', 'UNH', 'ULTA', 'UHS', 'UDR', 'UBER', 'UAL', 'TYL', 'TXT', 'TXN', 'TTWO', 'TTD', 'TT', 'TSN', 'TSLA', 'TSCO', 'TRV', 'TROW', 'TRMB', 'TRGP', 'TPR', 'TPL', 'TMUS', 'TMO', 'TKO', 'TJX', 'TGT', 'TFC', 'TER', 'TEL', 'TECH', 'TDY', 'TDG', 'TAP', 'T', 'SYY', 'SYK', 'SYF', 'SWKS', 'SWK', 'SW', 'STZ', 'STX', 'STT', 'STLD', 'STE', 'SRE', 'SPGI', 'SPG', 'SOLV', 'SO', 'SNPS', 'SNDK', 'SNA', 'SMCI', 'SLB', 'SJM', 'SHW', 'SCHW', 'SBUX', 'SBAC', 'SATS', 'RVTY', 'RTX', 'RSG', 'ROST', 'ROP', 'ROL', 'ROK', 'RMD', 'RL', 'RJF', 'RF', 'REGN', 'REG', 'RCL', 'QCOM', 'Q', 'PYPL', 'PWR', 'PTC', 'PSX', 'PSKY', 'PSA', 'PRU', 'PPL', 'PPG', 'POOL', 'PODD', 'PNW', 'PNR', 'PNC', 'PM', 'PLTR', 'PLD', 'PKG', 'PHM', 

In [30]:
removed = sp500.loc[
      (sp500["action"] == "removed") &
      (pd.to_datetime(sp500["date"]) >= "2000-01-01"),
      "ticker"
  ].unique()
print(len(removed))

597


In [31]:
def fetch_chunked(tickers, table, lib, mapping):
    """Download ticker history in chunks to avoid oversized API requests."""
    chunks = [tickers[i:i + CHUNK_SIZE] for i in range(0, len(tickers), CHUNK_SIZE)]
    existing = set(lib.list_symbols())
    failed_chunks = []

    print(f"Fetching {len(tickers)} tickers from {table}. {len(existing)} already in lib so we are skipping")

    for chunk in tqdm(chunks):
        todo = [t for t in chunk if t not in existing]

        if not todo:
            continue
        df = None
        for attempt in range(RETRY_ATTEMPTS):
            try:
                df = ndl.get_table(
                    table,
                    ticker=",".join(todo),
                    date={"gte": START, "lte": END},
                    paginate=True,
                )

                break
            except Exception as exc:
                if attempt ==RETRY_ATTEMPTS-1:
                    failed_chunks.append((todo, str(exc)))
                else:
                    time.sleep(2 ** attempt)

        if df is None or df.empty:
            continue

        df["date"] = pd.to_datetime(df["date"])

        for tkr, sub in df.groupby("ticker", sort=False):
            sub = (sub.drop(columns="ticker").set_index("date").sort_index())
            lib.write(tkr, sub, metadata={"permaticker": mapping.get(tkr)})
            existing.add(tkr)

        del df
        time.sleep(DELAY_BETWEEN_CHUNKS)
    print(f"done , {sum( len(c) for c, _ in failed_chunks)} failed")
    return failed_chunks

In [257]:
stock_prices = fetch_chunked(current_members, "SHARADAR/SEP", stocks_lib, stock_to_perma)

Fetching 503 tickers from SHARADAR/SEP. 0 already in lib so we are skipping


100%|██████████| 51/51 [06:11<00:00,  7.29s/it]

done , 0 failed


In [84]:
etf_list = ["XLK", "XLF", "XLV", "XLE", "XLI", "XLY", "XLP", "XLB", "XLU", "XLRE", "XLC", "SPY"]
etf_prices = fetch_chunked(etf_list, "SHARADAR/SFP", etf_lib, etf_to_perma)

Fetching 12 tickers from SHARADAR/SFP. 11 already in lib so we are skipping


100%|██████████| 2/2 [00:01<00:00,  1.81it/s]

done , 0 failed


In [271]:
removed_stocks = fetch_chunked(removed, "SHARADAR/SEP", removed_lib, stock_to_perma)

Fetching 597 tickers from SHARADAR/SEP. 0 already in lib so we are skipping


100%|██████████| 60/60 [04:59<00:00,  4.98s/it]

done , 0 failed


In [32]:
stocks.columns.to_list()

['table',
 'permaticker',
 'ticker',
 'name',
 'exchange',
 'isdelisted',
 'category',
 'cusips',
 'siccode',
 'sicsector',
 'sicindustry',
 'figi',
 'famaindustry',
 'sector',
 'industry',
 'scalemarketcap',
 'scalerevenue',
 'relatedtickers',
 'currency',
 'location',
 'lastupdated',
 'firstadded',
 'firstpricedate',
 'lastpricedate',
 'firstquarter',
 'lastquarter',
 'secfilings',
 'companysite']

Adding the metadata of price start date, price end date, and the permaticker for all the ticker in the lib.

In [33]:
def add_metadata(lib, source_df, fields=None, overwrite=False):
    if fields is None:
        fields = source_df.columns.to_list()

    clean = source_df[fields].where(source_df[fields].notna(), None)
    info_map = clean.to_dict("index")

    updated = missing = unchanged = 0

    for sym in tqdm(lib.list_symbols(), desc="Adding metadata"):
        info = info_map.get(sym)
        if info is None:
            missing += 1
            continue

        existing = lib.read_metadata(sym).metadata or {}
        new_meta = dict (existing)
        for k, v in info.items():
            if overwrite or k not in new_meta:
                new_meta[k] = v

        if new_meta == existing:
            unchanged += 1
            continue

        lib.write_metadata(sym, new_meta)
        updated +=1

    print(f'Updated {updated} symbols, {unchanged} unchanged symbols, {missing} missing symbols')
    return {"updated": updated, "unchanged": unchanged, "missing": missing}

In [34]:
stocks_meta = (stocks.loc[stocks["table"] == "SEP",
                                ["ticker", "industry","permaticker", "sector", "firstpricedate" , "lastpricedate"]]
                                .drop_duplicates("ticker")
                                .set_index("ticker"))
print(stocks_meta.head())
add_metadata(stocks_lib, stocks_meta, fields=["sector", "industry", "firstpricedate", "lastpricedate"])

               industry  permaticker       sector firstpricedate lastpricedate
ticker                                                                        
QADRU              None      6399211         None     2026-05-20    2026-05-21
RACC    Shell Companies      6399210  Industrials     2026-05-20    2026-05-21
VECA.U  Shell Companies      6399207  Industrials     2026-05-19    2026-05-21
RREVU              None      6399206         None     2026-05-20    2026-05-21
OPTH               None      6399205         None     2026-05-20    2026-05-21


Adding metadata: 100%|██████████| 503/503 [00:00<00:00, 17754.38it/s]

Updated 0 symbols, 503 unchanged symbols, 0 missing symbols


{'updated': 0, 'unchanged': 503, 'missing': 0}

In [35]:
removed_stocks_meta = (stocks.loc[stocks["table"] == "SEP",
                                ["ticker", "permaticker","industry", "sector", "firstpricedate" , "lastpricedate"]]
                                .drop_duplicates("ticker")
                                .set_index("ticker"))
print(removed_stocks_meta.head())
add_metadata(removed_lib, removed_stocks_meta, fields=["sector","industry", "firstpricedate", "lastpricedate"])

        permaticker         industry       sector firstpricedate lastpricedate
ticker                                                                        
QADRU       6399211             None         None     2026-05-20    2026-05-21
RACC        6399210  Shell Companies  Industrials     2026-05-20    2026-05-21
VECA.U      6399207  Shell Companies  Industrials     2026-05-19    2026-05-21
RREVU       6399206             None         None     2026-05-20    2026-05-21
OPTH        6399205             None         None     2026-05-20    2026-05-21


Adding metadata: 100%|██████████| 596/596 [00:00<00:00, 22071.58it/s]

Updated 0 symbols, 596 unchanged symbols, 0 missing symbols


{'updated': 0, 'unchanged': 596, 'missing': 0}

In [36]:
def build_sp500_intervals(sp500):
  """events → (ticker, start, end) intervals. end is NaT if still in."""
  df = sp500.copy()
  df["date"] = pd.to_datetime(df["date"])
  df = df.sort_values(["ticker", "date"])

  rows = []
  for ticker, grp in df.groupby("ticker"):
      events = grp.loc[grp["action"].isin(["added", "removed"])]
      historical = grp.loc[grp["action"] == "historical", "date"]
      has_current = (grp["action"] == "current").any()

      open_start = None
      for _, ev in events.iterrows():
          if ev["action"] == "added":
              if open_start is None:
                  open_start = ev["date"]
          else:  # removed
              if open_start is not None:
                  start = open_start
              else:
                  prior = historical[historical < ev["date"]]
                  start = prior.min() if not prior.empty else pd.NaT
              rows.append((ticker, start, ev["date"]))
              open_start = None

      if open_start is not None:
          rows.append((ticker, open_start, pd.NaT))
      elif has_current and events.empty:
          # In the index since before tracked events
          start = historical.min() if not historical.empty else pd.NaT
          rows.append((ticker, start, pd.NaT))

  return pd.DataFrame(rows, columns=["ticker", "start", "end"])

intervals = build_sp500_intervals(sp500)
print(f"{len(intervals)} intervals built")
print(intervals.head())

1237 intervals built
  ticker      start        end
0      A 2000-06-05        NaT
1    AAL 2015-03-17 2024-09-23
2  AAMRQ 1957-03-04 2003-03-14
3    AAP 2015-07-07 2023-08-25
4   AAPL 1982-11-30        NaT


In [37]:
def lookup_permaticker(ticker, stock_to_perma, stocks_lib, removed_lib):
  if ticker in stock_to_perma:
      return stock_to_perma[ticker]
  for lib in (stocks_lib, removed_lib):
      if ticker in lib.list_symbols():
          meta = lib.read_metadata(ticker).metadata or {}
          if "permaticker" in meta:
              return meta["permaticker"]
  return None

intervals["permaticker"] = intervals["ticker"].apply(
  lambda t: lookup_permaticker(t, stock_to_perma, stocks_lib, removed_lib)
)
unmapped = intervals[intervals["permaticker"].isna()]
print(f"{len(unmapped)} intervals failed to map; sample:")
print(unmapped.head())

25 intervals failed to map; sample:
    ticker      start        end  permaticker
19     AD2 1957-03-04 1986-05-08          NaN
168   BRNO 1991-05-21 1995-08-17          NaN
175    BV2 1991-01-09 1994-09-29          NaN
196   CBB1 1976-07-01 1998-01-27          NaN
230   CGW2 1957-03-04 1966-12-15          NaN


In [38]:
open_now = intervals["end"].isna().sum()
print(f"open intervals: {open_now}  (expect ~503)")

open intervals: 503  (expect ~503)


In [39]:
def show(ticker):
  return intervals.loc[intervals["ticker"] == ticker]
print(show("LEH"))    # end ≈ 2008-09-15
print(show("ENRN"))   # end ≈ 2001-11-28
print(show("SIVB"))   # end ≈ 2023-03-15
print(show("META"))   # open interval
print(show("FB"))     # may be empty if Sharadar consolidated under META

Empty DataFrame
Columns: [ticker, start, end, permaticker]
Index: []
Empty DataFrame
Columns: [ticker, start, end, permaticker]
Index: []
Empty DataFrame
Columns: [ticker, start, end, permaticker]
Index: []
    ticker      start end  permaticker
725   META 2013-12-20 NaT     194817.0
Empty DataFrame
Columns: [ticker, start, end, permaticker]
Index: []


In [40]:
def has_overlaps(grp):
  grp = grp.sort_values("start", na_position="first")
  ends = grp["end"].fillna(pd.Timestamp.max)
  starts = grp["start"].fillna(pd.Timestamp.min).shift(-1)
  return ((starts < ends) & starts.notna()).any()
overlap_count = intervals.groupby("permaticker").apply(has_overlaps).sum()
print(f"permatickers with overlapping intervals: {overlap_count}  (expect 0)")

permatickers with overlapping intervals: 0  (expect 0)


/tmp/ipykernel_867155/2341419665.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  overlap_count = intervals.groupby("permaticker").apply(has_overlaps).sum()


In [41]:
def members_on(intervals, date):
  d = pd.Timestamp(date)
  in_range = (intervals["start"].fillna(pd.Timestamp.min) <= d) & \
             (intervals["end"].fillna(pd.Timestamp.max) >= d)
  return intervals.loc[in_range, "permaticker"].nunique()

for year in range(2000, 2027):
  print(year, members_on(intervals, f"{year}-06-30"))

2000 500
2001 500
2002 500
2003 500
2004 501
2005 500
2006 500
2007 500
2008 502
2009 502
2010 501
2011 501
2012 500
2013 501
2014 502
2015 504
2016 508
2017 505
2018 505
2019 505
2020 505
2021 505
2022 503
2023 503
2024 503
2025 503
2026 503


In [42]:
if "metadata_lib" not in ac.list_libraries():
  ac.create_library("metadata_lib")
metadata_lib = ac["metadata_lib"]

metadata_lib.write("sp500_membership", intervals, metadata={
  "source": "SHARADAR/SP500",
  "fetched": pd.Timestamp.now(),
  "rows": len(intervals),
})

VersionedItem(symbol='sp500_membership', library='metadata_lib', data=n/a, version=4, metadata={'source': 'SHARADAR/SP500', 'fetched': Timestamp('2026-05-22 11:35:55.535610'), 'rows': 1237}, host='LMDB(path=/home/sysadmin/PycharmProjects/PM/database/arctic)', timestamp=1779464155544488799)

In [43]:
sp500.loc[sp500["ticker"] == "META", ["date", "action"]].sort_values("date").head()

,date,action
None,,
26207,2013-12-20,added
25909,2013-12-31,historical
25405,2014-03-31,historical
24893,2014-06-30,historical
24384,2014-09-30,historical


In [44]:
window = intervals[intervals["end"].isna() | (intervals["end"] >= "2000-01-01")]
print(f"in-window intervals: {len(window)}")
print(f"in-window unmapped: {window['permaticker'].isna().sum()}  (expect 0)")

in-window intervals: 1114
in-window unmapped: 0  (expect 0)


In [45]:
print(show("LEHMQ"))   # end ≈ 2008-09-15
print(show("ENRNQ"))   # end ≈ 2001-11-28
print(show("SIVBQ"))   # end ≈ 2023-03-15

    ticker      start        end  permaticker
664  LEHMQ 1998-01-09 2008-09-22     188897.0
    ticker      start        end  permaticker
397  ENRNQ 1980-01-01 2001-11-30     124095.0
    ticker      start        end  permaticker
996  SIVBQ 2018-03-19 2023-03-10     198834.0


In [46]:
overlapping = (
  intervals.dropna(subset=["permaticker"])
  .groupby("permaticker", group_keys=False)
  .filter(has_overlaps)
  .sort_values(["permaticker", "start"])
)
print(overlapping)

Empty DataFrame
Columns: [ticker, start, end, permaticker]
Index: []


In [47]:
sym = "AAPL"
print(f"{sym} in lib?    ", sym in stocks_lib.list_symbols())
print(f"{sym} in source? ", sym in stocks_meta.index)

AAPL in lib?     True
AAPL in source?  True


In [48]:
fields = ["sector"]
clean = stocks_meta[fields].where(stocks_meta[fields].notna(), None)
info_map = clean.to_dict("index")

print("info_map size:", len(info_map))
print("AAPL in info_map:", "AAPL" in info_map)
print("info_map['AAPL']:", info_map.get("AAPL"))

# Now mimic the loop's first iteration
sample = next(iter(stocks_lib.list_symbols()))
print(f"\nfirst lib symbol: {sample!r}")
print(f"  type: {type(sample)}")
print(f"  in info_map: {sample in info_map}")
print(f"  info_map.get: {info_map.get(sample)}")

info_map size: 21804
AAPL in info_map: True
info_map['AAPL']: {'sector': 'Technology'}

first lib symbol: 'TECH'
  type: <class 'str'>
  in info_map: True
  info_map.get: {'sector': 'Healthcare'}


In [49]:
stocks.loc[stocks["table"] == "SEP", "sector"].value_counts(dropna=False)

sector
Industrials               4185
Technology                3673
Healthcare                3158
Financial Services        3078
Consumer Cyclical         1979
Communication Services    1209
Basic Materials           1076
Real Estate               1075
Energy                     961
Consumer Defensive         830
Utilities                  426
None                       154
Name: count, dtype: int64

In [50]:
sectors = (
    stocks.loc[stocks["table"] == "SEP", ["ticker", "sector", "industry"]]
        .drop_duplicates("ticker")
        .set_index("ticker")
)
sectors

,sector,industry
ticker,,
QADRU,None,None
RACC,Industrials,Shell Companies
VECA.U,Industrials,Shell Companies
RREVU,None,None
OPTH,None,None
...,...,...
BBUCQ,Consumer Cyclical,Restaurants
NWGN1,Technology,Software - Application
CYRS1,Communication Services,Telecom Services


In [51]:
current_counts = sectors.loc[sectors.index.intersection(current_members), "sector"].value_counts()
removed_counts = sectors.loc[sectors.index.intersection(removed), "sector"].value_counts()

In [52]:
current_counts

sector
Technology                85
Industrials               75
Financial Services        70
Healthcare                59
Consumer Cyclical         47
Consumer Defensive        35
Utilities                 32
Real Estate               31
Basic Materials           26
Communication Services    22
Energy                    21
Name: count, dtype: int64

In [53]:
removed_counts

sector
Technology                97
Consumer Cyclical         84
Financial Services        76
Healthcare                61
Industrials               55
Energy                    52
Consumer Defensive        45
Communication Services    44
Basic Materials           42
Utilities                 29
Real Estate               12
Name: count, dtype: int64

In [54]:
comparison = pd.concat([current_counts.rename("current"),
                          removed_counts.rename("removed")], axis=1).fillna(0).astype(int)
comparison["total"] = comparison["current"] + comparison["removed"]
print(comparison.sort_values("total", ascending=False))

                        current  removed  total
sector                                         
Technology                   85       97    182
Financial Services           70       76    146
Consumer Cyclical            47       84    131
Industrials                  75       55    130
Healthcare                   59       61    120
Consumer Defensive           35       45     80
Energy                       21       52     73
Basic Materials              26       42     68
Communication Services       22       44     66
Utilities                    32       29     61
Real Estate                  31       12     43


In [55]:
for tickers in removed:
    if tickers in stocks_lib.list_symbols():
        stocks_lib.delete(tickers)
        print(f'{tickers} removed. {len(stocks_lib.list_symbols())} number of existing symbols')

PCG removed. 502 number of existing symbols
EQT removed. 501 number of existing symbols
KDP removed. 500 number of existing symbols
FSLR removed. 499 number of existing symbols
JBL removed. 498 number of existing symbols
TER removed. 497 number of existing symbols
LDOS removed. 496 number of existing symbols
AMD removed. 495 number of existing symbols
TMUS removed. 494 number of existing symbols
CIEN removed. 493 number of existing symbols
TT removed. 492 number of existing symbols
TEL removed. 491 number of existing symbols
CB removed. 490 number of existing symbols
PTC removed. 489 number of existing symbols
OKE removed. 488 number of existing symbols
